In [0]:
%sql
SELECT
    c.customer_id,
    c.name,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(oi.quantity * oi.unit_price) AS lifetime_value
FROM test_catalog.bronze.bronze_customer  c
JOIN test_catalog.bronze.bronze_orders o
    ON c.customer_id = o.customer_id
JOIN test_catalog.bronze.bronze_order_items  oi
    ON o.order_id = oi.order_id
GROUP BY
    c.customer_id,
    c.name
ORDER BY lifetime_value DESC;

In [0]:
%sql
WITH order_totals AS (
    SELECT
        order_id,
        SUM(quantity * unit_price) AS order_amount
    FROM test_catalog.bronze.bronze_order_items
    GROUP BY order_id
),
customer_sales AS (
    SELECT
        o.customer_id,
        COUNT(*) AS total_orders,
        SUM(ot.order_amount) AS lifetime_value
    FROM test_catalog.bronze.bronze_orders o
    JOIN order_totals ot
        ON o.order_id = ot.order_id
    GROUP BY o.customer_id
)
SELECT
    c.customer_id,
    c.name,
    cs.total_orders,
    cs.lifetime_value
FROM customer_sales cs
JOIN test_catalog.bronze.bronze_customer c
    ON cs.customer_id = c.customer_id
ORDER BY cs.lifetime_value DESC;

In [0]:
%sql
DESCRIBE DETAIL test_catalog.bronze.bronze_customer;

In [0]:
%sql
delete from test_catalog.bronze.bronze_customer where customer_id='CUST02404'

In [0]:
%sql
insert into test_catalog.bronze.bronze_customer (customer_id, name, contact_mob_no) values('CUST00000','Customer_00001','3333333333');

In [0]:
%sql
delete from test_catalog.bronze.bronze_customer where customer_id='CUST01117'

In [0]:
%sql
describe history test_catalog.bronze.bronze_customer

# Overwrite Operation on test_catalog.bronze.bronze_customer

In [0]:
%sql
CREATE OR REPLACE VIEW test_catalog.default.customer_view AS
SELECT *
FROM test_catalog.bronze.bronze_customer;

In [0]:
df = spark.table("test_catalog.default.customer_view")

In [0]:
%sql
SELECT
    pr.region_name,
    p.product_name,
    SUM(oi.quantity) qty
FROM test_catalog.bronze.bronze_order_items oi
JOIN test_catalog.bronze.bronze_products p
    ON oi.product_id = p.product_id
JOIN test_catalog.bronze.bronze_product_region pr
    ON p.product_id = pr.product_id
GROUP BY
    pr.region_name,
    p.product_name
ORDER BY
    region_name,
    qty DESC;

In [0]:
%sql
EXPLAIN EXTENDED
SELECT
    pr.region_name,
    p.product_name,
    SUM(oi.quantity) qty
FROM test_catalog.bronze.bronze_order_items oi
JOIN test_catalog.bronze.bronze_products p
    ON oi.product_id = p.product_id
JOIN test_catalog.bronze.bronze_product_region pr
    ON p.product_id = pr.product_id
GROUP BY
    pr.region_name,
    p.product_name
ORDER BY
    region_name,
    qty DESC;

In [0]:
%sql

EXPLAIN EXTENDED
SELECT
customer_id,
COUNT(DISTINCT region_name)
FROM test_catalog.bronze.bronze_orders o
JOIN test_catalog.bronze.bronze_order_items oi
ON o.order_id=oi.order_id
JOIN test_catalog.bronze.bronze_product_region pr
ON oi.product_id=pr.product_id
GROUP BY customer_id
HAVING COUNT(DISTINCT region_name)>3;